In [ ]:
%%bash
set -e

apt-get update -y
apt-get install -y xvfb x11-utils mesa-utils libgl1-mesa-glx libglib2.0-0
pip -q install ai2thor tqdm

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
x11-utils is already the newest version (7.7+5build2).
mesa-utils is already the newest version (8.4.0-1ubuntu1).
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!rm -f /content/config.py /content/config*.py
!rm -rf /content/pilot_full

In [ ]:
import os
import shutil
import time
from google.colab import files, drive



def ensure_google_drive_mounted(mount_root="/content/drive"):
    """
    Ensure Google Drive is correctly mounted before any Step1 output is written.

    This avoids the dangerous case where /content/drive/MyDrive exists
    as a local Colab directory but is not actually Google Drive.
    """

    my_drive = os.path.join(mount_root, "MyDrive")

    if os.path.ismount(mount_root) and os.path.exists(my_drive):
        print("Google Drive is already mounted at:", mount_root)
        return

    if os.path.exists(mount_root) and os.listdir(mount_root):
        backup_root = f"/content/drive_stale_backup_{int(time.time())}"

        print("=" * 80)
        print("WARNING: /content/drive exists and is non-empty,")
        print("but it is not recognized as a mounted Google Drive.")
        print("This may be stale local runtime data.")
        print("Moving it aside to avoid writing Step1 outputs to local disk.")
        print("Stale directory will be moved to:", backup_root)
        print("=" * 80)

        shutil.move(mount_root, backup_root)

    os.makedirs(mount_root, exist_ok=True)

    if os.listdir(mount_root):
        raise RuntimeError(
            f"{mount_root} is still not empty before mounting. "
            "Restart runtime before continuing."
        )


    drive.mount(mount_root, force_remount=True)

    if not os.path.exists(my_drive):
        raise RuntimeError(
            "Google Drive mount failed: /content/drive/MyDrive does not exist."
        )

    print("Google Drive mounted successfully at:", mount_root)


def verify_drive_project_root():
    """
    Verify that the intended Google Drive project root is visible and writable.
    """

    project_root = "/content/drive/MyDrive/Colab Notebooks/linear_probe_full"

    os.makedirs(project_root, exist_ok=True)

    print("Drive project root is visible and writable:")
    print(project_root)


ensure_google_drive_mounted()
verify_drive_project_root()



PILOT_ROOT = "/content/pilot_full"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")  # runtime placeholder only

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Make pilot_full importable as a Python package.
init_path = os.path.join(PILOT_ROOT, "__init__.py")
open(init_path, "a").close()


# Upload config

print("Please upload one config file:")
print("  - config.py")
print("  - config_positional_context_full.py")
print("  - config_positional_context_full_drive.py")

uploaded = files.upload()

allowed_config_names = [
    "config.py",
    "config_positional_context_full.py",
    "config_positional_context_full_drive.py",
]

uploaded_config_names = [
    name for name in allowed_config_names
    if name in uploaded
]

if len(uploaded_config_names) == 0:
    raise FileNotFoundError(
        "No valid config file was uploaded. "
        "Please upload one of: config.py, "
        "config_positional_context_full.py, "
        "config_positional_context_full_drive.py."
    )

if len(uploaded_config_names) > 1:
    raise ValueError(
        "Multiple config files were uploaded. "
        "Please upload only one config file."
    )

config_src_name = uploaded_config_names[0]
src = os.path.join("/content", config_src_name)
dst = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(src, dst)

print("Uploaded config source:", config_src_name)
print("Copied runtime config to:", dst)
print("Project root:", PILOT_ROOT)
print("Scripts dir:", SCRIPTS_DIR)
print("Runtime data dir:", DATA_DIR)

Google Drive is already mounted at: /content/drive
Drive project root is visible and writable:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full
Please upload one config file:
  - config.py
  - config_positional_context_full.py
  - config_positional_context_full_drive.py


Saving config_positional_context_full_drive.py to config_positional_context_full_drive.py
Uploaded config source: config_positional_context_full_drive.py
Copied runtime config to: /content/pilot_full/config.py
Project root: /content/pilot_full
Scripts dir: /content/pilot_full/scripts
Runtime data dir: /content/pilot_full/data


In [ ]:
import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

print("Loaded config from: /content/pilot_full/config.py")

print("\nProject paths:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("SCRIPTS_DIR:", cfg.SCRIPTS_DIR)
print("DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("DATA_DIR:", cfg.DATA_DIR)
print("STEP1_OUTPUT_DIR:", cfg.STEP1_OUTPUT_DIR)

print("\nScenes:")
print("Number of scenes:", len(cfg.SCENES))
print("First scenes:", cfg.SCENES[:5])
print("Last scenes:", cfg.SCENES[-5:])

print("\nController config:")
print("CONTROLLER_WIDTH:", cfg.CONTROLLER_WIDTH)
print("CONTROLLER_HEIGHT:", cfg.CONTROLLER_HEIGHT)
print("AGENT_MODE:", cfg.AGENT_MODE)

print("\nStep 1 config:")
print("USE_PICKUPABLE_ONLY:", cfg.USE_PICKUPABLE_ONLY)
print("EXCLUDED_TYPES:", cfg.EXCLUDED_TYPES)
print("ANCHOR_TYPES:", cfg.ANCHOR_TYPES)
print("STRUCTURAL_TYPES:", cfg.STRUCTURAL_TYPES)
print("GRID_NX:", cfg.GRID_NX)
print("GRID_NZ:", cfg.GRID_NZ)
print("PARENT_COMPLETION_RADIUS:", cfg.PARENT_COMPLETION_RADIUS)
print("CHILD_COMPLETION_RADIUS:", cfg.CHILD_COMPLETION_RADIUS)
print("MAX_CHILDREN_PER_ANCHOR:", cfg.MAX_CHILDREN_PER_ANCHOR)
print("NEAR_GRAPH_THRESHOLD:", cfg.NEAR_GRAPH_THRESHOLD)
print("MIN_OBJECTS_PER_CLUSTER:", cfg.MIN_OBJECTS_PER_CLUSTER)
print("MAX_OBJECTS_PER_CLUSTER:", cfg.MAX_OBJECTS_PER_CLUSTER)
print("MIN_ANCHORS_PER_CLUSTER:", cfg.MIN_ANCHORS_PER_CLUSTER)
print("MIN_SMALL_OBJECTS_PER_CLUSTER:", cfg.MIN_SMALL_OBJECTS_PER_CLUSTER)
print("MAX_CLUSTERS_PER_CHUNK:", cfg.MAX_CLUSTERS_PER_CHUNK)
print("KEEP_WEAK_CLUSTERS:", cfg.KEEP_WEAK_CLUSTERS)


import os

print("\nStep1 output path verification:")
print("STEP1_OUTPUT_DIR:", cfg.STEP1_OUTPUT_DIR)

if not cfg.STEP1_OUTPUT_DIR.startswith("/content/drive/MyDrive/"):
    raise RuntimeError(
        f"STEP1_OUTPUT_DIR is not under Google Drive: {cfg.STEP1_OUTPUT_DIR}"
    )

os.makedirs(cfg.STEP1_OUTPUT_DIR, exist_ok=True)

step1_test_file = os.path.join(cfg.STEP1_OUTPUT_DIR, "_step1_output_dir_write_test.txt")

with open(step1_test_file, "w", encoding="utf-8") as f:
    f.write("test\n")

if not os.path.exists(step1_test_file):
    raise RuntimeError("STEP1_OUTPUT_DIR write test failed.")

os.remove(step1_test_file)

print("STEP1_OUTPUT_DIR write test passed.")

Loaded config from: /content/pilot_full/config.py

Project paths:
PILOT_ROOT: /content/pilot_full
SCRIPTS_DIR: /content/pilot_full/scripts
DRIVE_ROOT: /content/drive/MyDrive/Colab Notebooks/linear_probe_full
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data
STEP1_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt

Scenes:
Number of scenes: 120
First scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5']
Last scenes: ['FloorPlan426', 'FloorPlan427', 'FloorPlan428', 'FloorPlan429', 'FloorPlan430']

Controller config:
CONTROLLER_WIDTH: 300
CONTROLLER_HEIGHT: 300
AGENT_MODE: default

Step 1 config:
USE_PICKUPABLE_ONLY: False
EXCLUDED_TYPES: {'Floor', 'Wall', 'Ceiling'}
ANCHOR_TYPES: {'SinkBasin', 'Desk', 'StoveBurner', 'Cabinet', 'Drawer', 'Chair', 'GarbageCan', 'BathtubBasin', 'Sink', 'Toilet', 'Fridge', 'DiningTable', 'ToiletPaperHanger', 'SideTable', 'CoffeeTable', 'Shelf', 'HandTowelHolder', 'Microwave', 'Bathtub', 

In [ ]:
%%writefile /content/pilot_full/scripts/step1_extract_chunk_ground_truth.py

import json
import math
import os
import random
import sys
import time
from collections import deque
from typing import Any, Dict, List, Tuple, Set

from tqdm import tqdm
from ai2thor.controller import Controller


CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PILOT_DIR = os.path.dirname(CURRENT_DIR)
PROJECT_ROOT = os.path.dirname(PILOT_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


from pilot_full.config import (
    STEP1_OUTPUT_DIR,
    SCENES,
    RANDOM_SEED,
    USE_PICKUPABLE_ONLY,
    EXCLUDED_TYPES,
    ANCHOR_TYPES,
    STRUCTURAL_TYPES,
    GRID_NX,
    GRID_NZ,
    PARENT_COMPLETION_RADIUS,
    CHILD_COMPLETION_RADIUS,
    MAX_CHILDREN_PER_ANCHOR,
    NEAR_GRAPH_THRESHOLD,
    MIN_OBJECTS_PER_CLUSTER,
    MAX_OBJECTS_PER_CLUSTER,
    MIN_ANCHORS_PER_CLUSTER,
    MIN_SMALL_OBJECTS_PER_CLUSTER,
    MAX_CLUSTERS_PER_CHUNK,
    KEEP_WEAK_CLUSTERS,
    CONTROLLER_WIDTH,
    CONTROLLER_HEIGHT,
    AGENT_MODE,
    )


random.seed(RANDOM_SEED)


# Basic IO helpers

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def save_json(path: str, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def progress_log_path() -> str:
    return os.path.join(STEP1_OUTPUT_DIR, "_step1_progress.log")


def write_progress(message: str) -> None:
    """
    Append one progress line to a log file under STEP1_OUTPUT_DIR.

    This is more reliable than relying only on Colab cell stdout,
    because %%bash/xvfb-run output may not refresh in real time.
    """
    ensure_dir(STEP1_OUTPUT_DIR)

    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp}\t{message}\n"

    with open(progress_log_path(), "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        os.fsync(f.fileno())

    print(message, flush=True)


# AI2-THOR controller helpers

def create_controller(initial_scene: str) -> Controller:
    write_progress("=" * 80)
    write_progress(f"CONTROLLER_INIT_START\t{initial_scene}")
    write_progress(
        "CONTROLLER_FLAGS\t"
        "renderDepthImage=False\t"
        "renderInstanceSegmentation=False\t"
        "renderSemanticSegmentation=False"
    )
    write_progress(
        f"CONTROLLER_CONFIG\t"
        f"width={CONTROLLER_WIDTH}\t"
        f"height={CONTROLLER_HEIGHT}\t"
        f"agentMode={AGENT_MODE}\t"
        f"timeout=300"
    )

    controller = Controller(
        scene=initial_scene,
        width=CONTROLLER_WIDTH,
        height=CONTROLLER_HEIGHT,
        agentMode=AGENT_MODE,
        renderDepthImage=False,
        renderInstanceSegmentation=False,
        renderSemanticSegmentation=False,
        timeout=300,
    )

    write_progress(f"CONTROLLER_INIT_DONE\t{initial_scene}")
    write_progress("=" * 80)
    return controller

def safe_stop_controller(controller: Controller) -> None:
    """
    Stop a controller without crashing the whole script.
    """
    try:
        write_progress("CONTROLLER_STOP_START")
        controller.stop()
        write_progress("CONTROLLER_STOP_DONE")
    except Exception as stop_error:
        write_progress(
            f"CONTROLLER_STOP_ERROR\t{repr(stop_error)}"
        )

def restart_controller_after_failure(
    controller: Controller,
    next_scene: str,
) -> Controller:
    """
    Restart AI2-THOR after a scene failure.

    This prevents one failed scene from corrupting all later scenes.
    """
    write_progress(
        f"CONTROLLER_RESTART_START\tnext_scene={next_scene}"
    )

    safe_stop_controller(controller)

    write_progress("CONTROLLER_RESTART_SLEEP_START\tseconds=2")
    time.sleep(2)
    write_progress("CONTROLLER_RESTART_SLEEP_DONE")

    write_progress(
        f"CONTROLLER_RESTART_CREATE\tnext_scene={next_scene}"
    )
    new_controller = create_controller(next_scene)

    write_progress(
        f"CONTROLLER_RESTART_DONE\tnext_scene={next_scene}"
    )
    return new_controller

# Extract object metadata used by later experiments.
#
# Geometry fields such as position and bbox are kept for Experiment B.
# Receptacle links are kept for Experiment A.

def get_position(obj: Dict[str, Any]) -> Dict[str, float]:
    pos = obj.get("position", {}) or {}
    return {
        "x": float(pos.get("x", 0.0)),
        "y": float(pos.get("y", 0.0)),
        "z": float(pos.get("z", 0.0)),
    }


def get_bbox(obj: Dict[str, Any]) -> Dict[str, Any]:
    aabb = obj.get("axisAlignedBoundingBox", {}) or {}
    center = aabb.get("center", {}) or {}
    size = aabb.get("size", {}) or {}

    return {
        "center": {
            "x": float(center.get("x", 0.0)),
            "y": float(center.get("y", 0.0)),
            "z": float(center.get("z", 0.0)),
        },
        "size": {
            "x": float(size.get("x", 0.0)),
            "y": float(size.get("y", 0.0)),
            "z": float(size.get("z", 0.0)),
        },
        "cornerPoints": aabb.get("cornerPoints", []) or [],
    }


def get_rotation(obj: Dict[str, Any]) -> Dict[str, float]:
    rot = obj.get("rotation", {}) or {}
    return {
        "x": float(rot.get("x", 0.0)),
        "y": float(rot.get("y", 0.0)),
        "z": float(rot.get("z", 0.0)),
    }


def is_valid_object(obj: Dict[str, Any]) -> bool:

    # Keep pickupable objects plus receptacle / structural objects.
    # Do not restrict to pickupable objects only.
    if "objectType" not in obj or "position" not in obj:
        return False

    obj_type = obj.get("objectType", "")

    if obj_type in EXCLUDED_TYPES:
        return False

    if USE_PICKUPABLE_ONLY and not obj.get("pickupable", False):
        return False

    if obj.get("objectId") is None:
        return False

    return True


def build_object_record(obj: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "objectId": obj.get("objectId"),
        "objectType": obj.get("objectType"),
        "position": get_position(obj),
        "bbox": get_bbox(obj),
        "rotation": get_rotation(obj),
        "pickupable": bool(obj.get("pickupable", False)),
        "receptacle": bool(obj.get("receptacle", False)),
        "openable": bool(obj.get("openable", False)),
        "visible": bool(obj.get("visible", False)),
        "parentReceptacles": obj.get("parentReceptacles") or [],
        "receptacleObjectIds": obj.get("receptacleObjectIds") or [],
    }


def build_object_index(objects: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {
        obj["objectId"]: obj
        for obj in objects
        if obj.get("objectId") is not None
    }


# Geometry helpers

def horizontal_distance(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> float:
    # Use x-z distance to preserve local floor-plane coherence.
    # y is height and should not be used for chunk splitting.
    pa = obj_a.get("position", {}) or {}
    pb = obj_b.get("position", {}) or {}

    dx = float(pa.get("x", 0.0)) - float(pb.get("x", 0.0))
    dz = float(pa.get("z", 0.0)) - float(pb.get("z", 0.0))
    return math.sqrt(dx * dx + dz * dz)


def compute_scene_bounds(objects: List[Dict[str, Any]]) -> Dict[str, float]:
    xs = [obj["position"]["x"] for obj in objects]
    zs = [obj["position"]["z"] for obj in objects]

    if not xs or not zs:
        return {
            "x_min": 0.0, "x_max": 1.0,
            "z_min": 0.0, "z_max": 1.0,
        }

    x_min, x_max = min(xs), max(xs)
    z_min, z_max = min(zs), max(zs)

    if abs(x_max - x_min) < 1e-6:
        x_min -= 0.5
        x_max += 0.5

    if abs(z_max - z_min) < 1e-6:
        z_min -= 0.5
        z_max += 0.5

    return {
        "x_min": float(x_min),
        "x_max": float(x_max),
        "z_min": float(z_min),
        "z_max": float(z_max),
    }


def get_chunk_bounds(
    scene_bounds: Dict[str, float],
    grid_i: int,
    grid_j: int,
    grid_nx: int,
    grid_nz: int,
) -> Dict[str, float]:
    x_min = scene_bounds["x_min"]
    x_max = scene_bounds["x_max"]
    z_min = scene_bounds["z_min"]
    z_max = scene_bounds["z_max"]

    x_step = (x_max - x_min) / grid_nx
    z_step = (z_max - z_min) / grid_nz

    return {
        "x_min": x_min + grid_i * x_step,
        "x_max": x_min + (grid_i + 1) * x_step,
        "z_min": z_min + grid_j * z_step,
        "z_max": z_min + (grid_j + 1) * z_step,
    }


def assign_chunk(
    obj: Dict[str, Any],
    scene_bounds: Dict[str, float],
    grid_nx: int,
    grid_nz: int,
) -> Tuple[int, int]:
    x = obj["position"]["x"]
    z = obj["position"]["z"]

    x_range = max(scene_bounds["x_max"] - scene_bounds["x_min"], 1e-6)
    z_range = max(scene_bounds["z_max"] - scene_bounds["z_min"], 1e-6)

    x_ratio = (x - scene_bounds["x_min"]) / x_range
    z_ratio = (z - scene_bounds["z_min"]) / z_range

    grid_i = int(x_ratio * grid_nx)
    grid_j = int(z_ratio * grid_nz)

    grid_i = min(grid_nx - 1, max(0, grid_i))
    grid_j = min(grid_nz - 1, max(0, grid_j))

    return grid_i, grid_j


# Object type helpers

def is_anchor_object(obj: Dict[str, Any]) -> bool:
    # Principle:
    # An anchor is a local organizer, not the only object in the chunk.
    return bool(obj.get("receptacle", False)) or obj.get("objectType") in ANCHOR_TYPES


def is_structural_object(obj: Dict[str, Any]) -> bool:
    return obj.get("objectType") in STRUCTURAL_TYPES


def is_small_object(obj: Dict[str, Any]) -> bool:
    if obj.get("pickupable", False):
        return True
    if not is_anchor_object(obj) and not is_structural_object(obj):
        return True
    return False


# Step 1A: initial x-z chunking.
# At this stage, each chunk is only a candidate local region,
# not a final paragraph unit.

def build_initial_chunks(
    scene: str,
    objects: List[Dict[str, Any]],
    scene_bounds: Dict[str, float],
) -> List[Dict[str, Any]]:
    grouped: Dict[Tuple[int, int], List[Dict[str, Any]]] = {}

    for obj in objects:
        grid_i, grid_j = assign_chunk(
            obj=obj,
            scene_bounds=scene_bounds,
            grid_nx=GRID_NX,
            grid_nz=GRID_NZ,
        )
        grouped.setdefault((grid_i, grid_j), []).append(obj)

    chunks = []

    for grid_i in range(GRID_NX):
        for grid_j in range(GRID_NZ):
            raw_objects = grouped.get((grid_i, grid_j), [])
            chunk_id = f"{scene}_chunk_{grid_i}_{grid_j}"
            bounds = get_chunk_bounds(
                scene_bounds=scene_bounds,
                grid_i=grid_i,
                grid_j=grid_j,
                grid_nx=GRID_NX,
                grid_nz=GRID_NZ,
            )

            chunks.append({
                "chunk_id": chunk_id,
                "grid_i": grid_i,
                "grid_j": grid_j,
                "bounds": bounds,
                "raw_object_ids": [
                    obj["objectId"] for obj in raw_objects
                ],
                "raw_objects": raw_objects,
            })

    return chunks


# Step 1B: parent / child completion.
#
# Before paragraph construction, fill in missing local relations:
# add the parent for included child objects, and add nearby children
# for included anchors when possible.

def add_object_by_id(
    selected: Dict[str, Dict[str, Any]],
    obj_id: str,
    object_index: Dict[str, Dict[str, Any]],
) -> bool:
    if obj_id in selected:
        return False

    obj = object_index.get(obj_id)
    if obj is None:
        return False

    selected[obj_id] = obj
    return True


def complete_parent_objects(
    selected: Dict[str, Dict[str, Any]],
    object_index: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    parent_added = []
    unresolved_parent_refs = []

    initial_objects = list(selected.values())

    for child in initial_objects:
        parent_ids = child.get("parentReceptacles", []) or []

        for parent_id in parent_ids:
            parent_obj = object_index.get(parent_id)

            if parent_obj is None:
                unresolved_parent_refs.append({
                    "child_id": child["objectId"],
                    "parent_id": parent_id,
                    "reason": "parent_not_in_object_index",
                })
                continue

            dist = horizontal_distance(child, parent_obj)

            if dist <= PARENT_COMPLETION_RADIUS:
                was_added = add_object_by_id(selected, parent_id, object_index)
                if was_added:
                    parent_added.append(parent_id)
            else:
                unresolved_parent_refs.append({
                    "child_id": child["objectId"],
                    "parent_id": parent_id,
                    "reason": "parent_too_far",
                    "horizontal_distance": dist,
                    "radius": PARENT_COMPLETION_RADIUS,
                })

    return {
        "parent_added": parent_added,
        "unresolved_parent_refs": unresolved_parent_refs,
    }


def complete_child_objects(
    selected: Dict[str, Dict[str, Any]],
    object_index: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    child_added = []

    anchor_objects = [
        obj for obj in list(selected.values())
        if is_anchor_object(obj)
    ]

    for anchor in anchor_objects:
        child_ids = anchor.get("receptacleObjectIds", []) or []
        child_objs = [
            object_index[cid]
            for cid in child_ids
            if cid in object_index
        ]

        child_objs.sort(key=lambda x: horizontal_distance(anchor, x))

        added_for_this_anchor = 0

        for child in child_objs:
            if added_for_this_anchor >= MAX_CHILDREN_PER_ANCHOR:
                break

            if horizontal_distance(anchor, child) > CHILD_COMPLETION_RADIUS:
                continue

            was_added = add_object_by_id(
                selected=selected,
                obj_id=child["objectId"],
                object_index=object_index,
            )

            if was_added:
                child_added.append(child["objectId"])
                added_for_this_anchor += 1

    return {
        "child_added": child_added
    }


def complete_chunk_objects(
    raw_objects: List[Dict[str, Any]],
    object_index: Dict[str, Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    selected = {
        obj["objectId"]: obj
        for obj in raw_objects
        if obj.get("objectId") is not None
    }

    num_raw = len(selected)

    parent_info = complete_parent_objects(selected, object_index)
    child_info = complete_child_objects(selected, object_index)

    completed = list(selected.values())

    diagnostics = {
        "num_raw_objects": num_raw,
        "num_after_completion": len(completed),
        "num_parent_added": len(parent_info["parent_added"]),
        "num_child_added": len(child_info["child_added"]),
        "parent_added_ids": parent_info["parent_added"],
        "child_added_ids": child_info["child_added"],
        "num_unresolved_parent_refs": len(parent_info["unresolved_parent_refs"]),
        "unresolved_parent_refs": parent_info["unresolved_parent_refs"],
    }

    return completed, diagnostics


# Step 1C: relation-potential graph.
#
# Use local relation edges to split incoherent chunks that were grouped
# only by x-z proximity. The graph is used for coherence checks, not as
# the final paragraph structure.

def build_relation_potential_edges(
    objects: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    object_ids = {obj["objectId"] for obj in objects}
    edges = []

    for obj in objects:
        child_id = obj["objectId"]
        for parent_id in obj.get("parentReceptacles", []) or []:
            if parent_id in object_ids:
                edges.append({
                    "source": child_id,
                    "target": parent_id,
                    "edge_type": "parent_child",
                    "strength": "strong",
                })

    n = len(objects)
    for i in range(n):
        for j in range(i + 1, n):
            obj_a = objects[i]
            obj_b = objects[j]
            dist = horizontal_distance(obj_a, obj_b)

            if dist <= NEAR_GRAPH_THRESHOLD:
                edges.append({
                    "source": obj_a["objectId"],
                    "target": obj_b["objectId"],
                    "edge_type": "near_xz",
                    "strength": "medium",
                    "horizontal_distance": dist,
                    "threshold": NEAR_GRAPH_THRESHOLD,
                })

    return edges


def connected_components(
    objects: List[Dict[str, Any]],
    edges: List[Dict[str, Any]],
) -> List[List[str]]:
    object_ids = [obj["objectId"] for obj in objects]
    adjacency: Dict[str, Set[str]] = {
        oid: set()
        for oid in object_ids
    }

    for edge in edges:
        a = edge["source"]
        b = edge["target"]
        if a in adjacency and b in adjacency:
            adjacency[a].add(b)
            adjacency[b].add(a)

    visited = set()
    components = []

    for oid in object_ids:
        if oid in visited:
            continue

        comp = []
        queue = deque([oid])
        visited.add(oid)

        while queue:
            cur = queue.popleft()
            comp.append(cur)

            for nxt in adjacency[cur]:
                if nxt not in visited:
                    visited.add(nxt)
                    queue.append(nxt)

        components.append(comp)

    components.sort(key=len, reverse=True)
    return components


# Step 1D: score anchors and trim each chunk.
#
# When a chunk has too many objects, keep the anchor objects first, followed by
# parent-child links and nearby small objects.

def score_anchor(
    anchor: Dict[str, Any],
    objects: List[Dict[str, Any]],
    edges: List[Dict[str, Any]],
) -> float:
    anchor_id = anchor["objectId"]

    num_children = len(anchor.get("receptacleObjectIds", []) or [])

    edge_count = 0
    for e in edges:
        if e["source"] == anchor_id or e["target"] == anchor_id:
            edge_count += 1

    nearby_small_count = 0
    for obj in objects:
        if obj["objectId"] == anchor_id:
            continue
        if is_small_object(obj) and horizontal_distance(anchor, obj) <= NEAR_GRAPH_THRESHOLD:
            nearby_small_count += 1

    receptacle_bonus = 1.0 if anchor.get("receptacle", False) else 0.0

    return (
        2.0 * num_children
        + 1.0 * edge_count
        + 1.0 * nearby_small_count
        + receptacle_bonus
    )


def select_top_anchors(
    objects: List[Dict[str, Any]],
    edges: List[Dict[str, Any]],
    top_k: int = 2,
) -> List[Dict[str, Any]]:
    anchors = [
        obj for obj in objects
        if is_anchor_object(obj)
    ]

    anchors.sort(
        key=lambda x: score_anchor(x, objects, edges),
        reverse=True,
    )

    return anchors[:top_k]


def truncate_cluster_objects(
    objects: List[Dict[str, Any]],
    edges: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    if len(objects) <= MAX_OBJECTS_PER_CLUSTER:
        return objects

    object_index = build_object_index(objects)
    selected_ids: List[str] = []

    def add_id(oid: str) -> None:
        if oid in object_index and oid not in selected_ids:
            selected_ids.append(oid)

    top_anchors = select_top_anchors(objects, edges, top_k=2)

    for anchor in top_anchors:
        add_id(anchor["objectId"])

    for edge in edges:
        if edge["edge_type"] == "parent_child":
            add_id(edge["source"])
            add_id(edge["target"])

    if top_anchors:
        primary_anchor = top_anchors[0]
        remaining = [
            obj for obj in objects
            if obj["objectId"] not in selected_ids
        ]
        remaining.sort(key=lambda x: horizontal_distance(primary_anchor, x))
    else:
        remaining = [
            obj for obj in objects
            if obj["objectId"] not in selected_ids
        ]
        remaining.sort(key=lambda x: x["objectType"])

    for obj in remaining:
        if len(selected_ids) >= MAX_OBJECTS_PER_CLUSTER:
            break
        add_id(obj["objectId"])

    return [
        object_index[oid]
        for oid in selected_ids[:MAX_OBJECTS_PER_CLUSTER]
    ]


# Step 1E: check whether each cluster is usable before building paragraphs.

def compute_cluster_quality(
    objects: List[Dict[str, Any]],
    edges: List[Dict[str, Any]],
) -> Dict[str, Any]:
    num_objects = len(objects)
    num_anchors = sum(1 for obj in objects if is_anchor_object(obj))
    num_small = sum(1 for obj in objects if is_small_object(obj))

    num_parent_child_edges = sum(
        1 for e in edges
        if e.get("edge_type") == "parent_child"
    )

    num_near_edges = sum(
        1 for e in edges
        if e.get("edge_type") == "near_xz"
    )

    keep = (
        num_objects >= MIN_OBJECTS_PER_CLUSTER
        and num_anchors >= MIN_ANCHORS_PER_CLUSTER
        and num_small >= MIN_SMALL_OBJECTS_PER_CLUSTER
    )

    return {
        "num_objects": num_objects,
        "num_anchor_objects": num_anchors,
        "num_small_objects": num_small,
        "num_relation_potential_edges": len(edges),
        "num_parent_child_edges": num_parent_child_edges,
        "num_near_edges": num_near_edges,
        "quality_status": "kept" if keep else "weak",
        "keep": keep,
    }


def build_clusters_from_chunk(
    scene: str,
    chunk: Dict[str, Any],
    object_index: Dict[str, Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    completed_objects, completion_diag = complete_chunk_objects(
        raw_objects=chunk["raw_objects"],
        object_index=object_index,
    )

    all_edges = build_relation_potential_edges(completed_objects)

    components = connected_components(
        objects=completed_objects,
        edges=all_edges,
    )

    completed_index = build_object_index(completed_objects)

    clusters = []
    num_components = len(components)

    for comp_idx, comp_ids in enumerate(components):
        comp_objects = [
            completed_index[oid]
            for oid in comp_ids
            if oid in completed_index
        ]

        comp_edges = [
            e for e in all_edges
            if e["source"] in comp_ids and e["target"] in comp_ids
        ]

        comp_objects = truncate_cluster_objects(
            objects=comp_objects,
            edges=comp_edges,
        )

        kept_ids = {obj["objectId"] for obj in comp_objects}
        comp_edges = [
            e for e in comp_edges
            if e["source"] in kept_ids and e["target"] in kept_ids
        ]

        quality = compute_cluster_quality(comp_objects, comp_edges)

        if not quality["keep"] and not KEEP_WEAK_CLUSTERS:
            continue

        top_anchors = select_top_anchors(comp_objects, comp_edges, top_k=2)

        cluster_id = f"{chunk['chunk_id']}_cluster_{comp_idx}"

        clusters.append({
            "cluster_id": cluster_id,
            "source_chunk_id": chunk["chunk_id"],
            "grid_i": chunk["grid_i"],
            "grid_j": chunk["grid_j"],
            "bounds": chunk["bounds"],
            "anchor_object_ids": [
                obj["objectId"] for obj in top_anchors
            ],
            "num_objects": len(comp_objects),
            "object_ids": [
                obj["objectId"] for obj in comp_objects
            ],
            "objects": comp_objects,
            "relation_potential_edges": comp_edges,
            "diagnostics": {
                **quality,
                "component_index": comp_idx,
                "component_size_before_truncation": len(comp_ids),
                "component_size_after_truncation": len(comp_objects),
            }
        })

        if len(clusters) >= MAX_CLUSTERS_PER_CHUNK:
            break

    chunk_diagnostics = {
        **completion_diag,
        "num_components_before_filter": num_components,
        "num_clusters_kept": len(clusters),
    }

    return clusters, chunk_diagnostics


# Main scene extraction

def extract_scene_ground_truth(controller: Controller, scene: str) -> Dict[str, Any]:
    event = controller.reset(scene=scene)
    metadata = event.metadata

    raw_objects = metadata.get("objects", []) or []

    valid_raw_objects = [
        obj for obj in raw_objects
        if is_valid_object(obj)
    ]

    object_records = [
        build_object_record(obj)
        for obj in valid_raw_objects
    ]

    object_index = build_object_index(object_records)

    scene_bounds = compute_scene_bounds(object_records)

    raw_chunks = build_initial_chunks(
        scene=scene,
        objects=object_records,
        scene_bounds=scene_bounds,
    )

    output_chunks = []
    total_clusters = 0
    total_cluster_objects = 0

    for chunk in raw_chunks:
        clusters, chunk_diag = build_clusters_from_chunk(
            scene=scene,
            chunk=chunk,
            object_index=object_index,
        )

        output_chunk = {
            "chunk_id": chunk["chunk_id"],
            "grid_i": chunk["grid_i"],
            "grid_j": chunk["grid_j"],
            "bounds": chunk["bounds"],
            "raw_object_ids": chunk["raw_object_ids"],
            "num_raw_objects": len(chunk["raw_object_ids"]),
            "num_clusters": len(clusters),
            "clusters": clusters,
            "diagnostics": chunk_diag,
        }

        output_chunks.append(output_chunk)
        total_clusters += len(clusters)
        total_cluster_objects += sum(c["num_objects"] for c in clusters)

    return {
        "scene": scene,
        "num_all_objects": len(raw_objects),
        "num_valid_objects": len(object_records),
        "scene_bounds": scene_bounds,
        "chunk_config": {
            "grid_nx": GRID_NX,
            "grid_nz": GRID_NZ,
            "parent_completion_radius": PARENT_COMPLETION_RADIUS,
            "child_completion_radius": CHILD_COMPLETION_RADIUS,
            "max_children_per_anchor": MAX_CHILDREN_PER_ANCHOR,
            "near_graph_threshold": NEAR_GRAPH_THRESHOLD,
            "min_objects_per_cluster": MIN_OBJECTS_PER_CLUSTER,
            "max_objects_per_cluster": MAX_OBJECTS_PER_CLUSTER,
            "min_anchors_per_cluster": MIN_ANCHORS_PER_CLUSTER,
            "min_small_objects_per_cluster": MIN_SMALL_OBJECTS_PER_CLUSTER,
            "max_clusters_per_chunk": MAX_CLUSTERS_PER_CHUNK,
            "keep_weak_clusters": KEEP_WEAK_CLUSTERS,
        },
        "num_chunks": len(output_chunks),
        "num_clusters": total_clusters,
        "total_cluster_object_mentions": total_cluster_objects,
        "chunks": output_chunks,
    }


# Main

def main() -> None:
    ensure_dir(STEP1_OUTPUT_DIR)

    write_progress("=" * 80)
    write_progress("START_STEP1")
    write_progress(f"OUTPUT_DIR\t{STEP1_OUTPUT_DIR}")
    write_progress(f"NUM_SCENES\t{len(SCENES)}")
    write_progress(f"GRID\t{GRID_NX}x{GRID_NZ}")
    write_progress(f"MAX_CLUSTERS_PER_CHUNK\t{MAX_CLUSTERS_PER_CHUNK}")
    write_progress(f"KEEP_WEAK_CLUSTERS\t{KEEP_WEAK_CLUSTERS}")
    write_progress(f"NEAR_GRAPH_THRESHOLD\t{NEAR_GRAPH_THRESHOLD}")
    write_progress(f"MIN_OBJECTS_PER_CLUSTER\t{MIN_OBJECTS_PER_CLUSTER}")
    write_progress(f"MAX_OBJECTS_PER_CLUSTER\t{MAX_OBJECTS_PER_CLUSTER}")
    write_progress("=" * 80)

    if not SCENES:
        raise ValueError("SCENES is empty. Please define at least one scene in config.py.")

    skip_existing_outputs = True

    failed_scenes = []
    controller = None

    try:
        write_progress(f"CONTROLLER_CREATE_INITIAL\t{SCENES[0]}")
        controller = create_controller(SCENES[0])
        write_progress(f"CONTROLLER_READY_INITIAL\t{SCENES[0]}")

        for scene_idx, scene in enumerate(
            tqdm(
                SCENES,
                desc="Extracting scenes",
                total=len(SCENES),
                unit="scene",
                dynamic_ncols=True,
            )
        ):
            scene_no = scene_idx + 1
            total_scenes = len(SCENES)
            out_path = os.path.join(STEP1_OUTPUT_DIR, f"{scene}.json")

            write_progress(
                f"SCENE_START\t{scene_no}/{total_scenes}\t{scene}"
            )

            if skip_existing_outputs and os.path.exists(out_path):
                write_progress(
                    f"SCENE_SKIP\t{scene_no}/{total_scenes}\t{scene}\t{out_path}"
                )
                continue

            try:
                write_progress(
                    f"SCENE_PROCESS\t{scene_no}/{total_scenes}\t{scene}"
                )

                scene_data = extract_scene_ground_truth(
                    controller=controller,
                    scene=scene,
                )

                save_json(out_path, scene_data)

                write_progress(
                    f"SCENE_SAVED\t{scene_no}/{total_scenes}\t{scene}\t"
                    f"all_objects={scene_data['num_all_objects']}\t"
                    f"valid_objects={scene_data['num_valid_objects']}\t"
                    f"chunks={scene_data['num_chunks']}\t"
                    f"clusters={scene_data['num_clusters']}\t"
                    f"path={out_path}"
                )

            except Exception as e:
                error_repr = repr(e)

                failed_scenes.append({
                    "scene": scene,
                    "error": error_repr,
                })

                write_progress(
                    f"SCENE_ERROR\t{scene_no}/{total_scenes}\t{scene}\t{error_repr}"
                )

                next_idx = scene_idx + 1

                if next_idx < len(SCENES):
                    next_scene = SCENES[next_idx]

                    write_progress(
                        f"CONTROLLER_RESTART_AFTER_ERROR\t"
                        f"failed_scene={scene}\tnext_scene={next_scene}"
                    )

                    try:
                        controller = restart_controller_after_failure(
                            controller=controller,
                            next_scene=next_scene,
                        )

                        write_progress(
                            f"CONTROLLER_RESTARTED\tnext_scene={next_scene}"
                        )

                    except Exception as restart_error:
                        restart_error_repr = repr(restart_error)

                        write_progress(
                            f"CONTROLLER_RESTART_FATAL\t"
                            f"failed_scene={scene}\t{restart_error_repr}"
                        )

                        failed_scenes.append({
                            "scene": f"controller_restart_after_{scene}",
                            "error": restart_error_repr,
                        })

                        break
                else:
                    write_progress(
                        f"NO_NEXT_SCENE_AFTER_ERROR\tfailed_scene={scene}"
                    )

                continue

    finally:
        if controller is not None:
            write_progress("CONTROLLER_STOP_FINAL")
            safe_stop_controller(controller)

    failed_path = os.path.join(STEP1_OUTPUT_DIR, "_failed_scenes_step1.json")
    save_json(failed_path, failed_scenes)

    write_progress("=" * 80)
    write_progress("STEP1_COMPLETED")
    write_progress(f"OUTPUT_DIR\t{STEP1_OUTPUT_DIR}")
    write_progress(f"FAILED_SCENE_LOG\t{failed_path}")
    write_progress(f"NUM_FAILED\t{len(failed_scenes)}")
    write_progress("=" * 80)

    print("=" * 80, flush=True)
    print("pilot_full Step 1 completed.", flush=True)
    print(f"Output dir: {STEP1_OUTPUT_DIR}", flush=True)
    print(f"Failed-scene log: {failed_path}", flush=True)

    if failed_scenes:
        print("\nFailed scenes:", flush=True)
        for item in failed_scenes:
            print(f"- {item['scene']}: {item['error']}", flush=True)

        print(
            "\nSome scenes failed. Successfully saved scene files remain usable. "
            "You can rerun the Step 1 cell; existing outputs will be skipped.",
            flush=True,
        )
    else:
        print("All scenes processed successfully.", flush=True)

    print("=" * 80, flush=True)

if __name__ == "__main__":
    main()

Writing /content/pilot_full/scripts/step1_extract_chunk_ground_truth.py


In [ ]:
%%bash
set +e

echo "Processes before cleanup:"
ps aux | grep -E "thor|Unity|Xvfb|xvfb" | grep -v grep || true

echo ""
echo "Killing possible stale AI2-THOR / Unity / Xvfb processes..."
pkill -f "thor-Linux" || true
pkill -f "AI2-THOR" || true
pkill -f "Unity" || true
pkill -f "Xvfb" || true
pkill -f "xvfb-run" || true

sleep 3

echo ""
echo "Processes after cleanup:"
ps aux | grep -E "thor|Unity|Xvfb|xvfb" | grep -v grep || true

echo "Cleanup done."

Processes before cleanup:

Killing possible stale AI2-THOR / Unity / Xvfb processes...

Processes after cleanup:
Cleanup done.


In [ ]:
%%bash
set -e

xvfb-run -a -s "-screen 0 1024x768x24" \
env LIBGL_ALWAYS_SOFTWARE=1 PYTHONUNBUFFERED=1 \
python -u /content/pilot_full/scripts/step1_extract_chunk_ground_truth.py

START_STEP1
OUTPUT_DIR	/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt
NUM_SCENES	120
GRID	3x3
MAX_CLUSTERS_PER_CHUNK	2
KEEP_WEAK_CLUSTERS	False
NEAR_GRAPH_THRESHOLD	0.8
MIN_OBJECTS_PER_CLUSTER	4
MAX_OBJECTS_PER_CLUSTER	15
CONTROLLER_CREATE_INITIAL	FloorPlan1
CONTROLLER_INIT_START	FloorPlan1
CONTROLLER_FLAGS	renderDepthImage=False	renderInstanceSegmentation=False	renderSemanticSegmentation=False
CONTROLLER_CONFIG	width=300	height=300	agentMode=default	timeout=300
CONTROLLER_INIT_DONE	FloorPlan1
CONTROLLER_READY_INITIAL	FloorPlan1
Extracting scenes:   0%|          | 0/120 [00:00<?, ?scene/s]SCENE_START	1/120	FloorPlan1
SCENE_PROCESS	1/120	FloorPlan1
SCENE_SAVED	1/120	FloorPlan1	all_objects=77	valid_objects=76	chunks=9	clusters=6	path=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt/FloorPlan1.json
Extracting scenes:   1%|          | 1/120 [00:04<09:52,  4.98s/scene]SCENE_START	2/120	FloorPlan2
SCENE_PROCESS	2/120	FloorPlan2
SCENE_SAVED	2/120	

In [ ]:
import os
import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

print("STEP1_OUTPUT_DIR:", cfg.STEP1_OUTPUT_DIR)

if not os.path.exists(cfg.STEP1_OUTPUT_DIR):
    raise FileNotFoundError(f"STEP1_OUTPUT_DIR does not exist: {cfg.STEP1_OUTPUT_DIR}")

json_files = sorted(
    name for name in os.listdir(cfg.STEP1_OUTPUT_DIR)
    if name.endswith(".json")
)

print("Number of Step 1 JSON files:", len(json_files))
print("Expected:", len(cfg.SCENES))
print("First files:", json_files[:5])
print("Last files:", json_files[-5:])

if len(json_files) != len(cfg.SCENES):
    print(f"Warning: expected {len(cfg.SCENES)} files, found {len(json_files)}")
else:
    print("Step 1 output count matches cfg.SCENES.")

STEP1_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt
Number of Step 1 JSON files: 115
Expected: 120
First files: ['FloorPlan1.json', 'FloorPlan10.json', 'FloorPlan11.json', 'FloorPlan12.json', 'FloorPlan13.json']
Last files: ['FloorPlan5.json', 'FloorPlan6.json', 'FloorPlan8.json', 'FloorPlan9.json', '_failed_scenes_step1.json']


In [ ]:
import os
import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

log_path = os.path.join(cfg.STEP1_OUTPUT_DIR, "_step1_progress.log")

print("Progress log:", log_path)

if not os.path.exists(log_path):
    print("Progress log not found.")
else:
    with open(log_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("Total log lines:", len(lines))
    print("\nLast 80 lines:")
    print("".join(lines[-80:]))

Progress log: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt/_step1_progress.log
Total log lines: 478

Last 80 lines:
2026-06-02 20:42:24	SCENE_PROCESS	102/120	FloorPlan412
2026-06-02 20:42:34	SCENE_SAVED	102/120	FloorPlan412	all_objects=35	valid_objects=34	chunks=9	clusters=3	path=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt/FloorPlan412.json
2026-06-02 20:42:34	SCENE_START	103/120	FloorPlan413
2026-06-02 20:42:34	SCENE_PROCESS	103/120	FloorPlan413
2026-06-02 20:42:40	SCENE_SAVED	103/120	FloorPlan413	all_objects=42	valid_objects=41	chunks=9	clusters=5	path=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt/FloorPlan413.json
2026-06-02 20:42:40	SCENE_START	104/120	FloorPlan414
2026-06-02 20:42:40	SCENE_PROCESS	104/120	FloorPlan414
2026-06-02 20:42:48	SCENE_SAVED	104/120	FloorPlan414	all_objects=40	valid_objects=39	chunks=9	clusters=4	path=/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt/FloorPla

In [ ]:
import os
import json
import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

if not cfg.SCENES:
    raise ValueError("cfg.SCENES is empty.")

example_scene = cfg.SCENES[0]
example_path = os.path.join(cfg.STEP1_OUTPUT_DIR, f"{example_scene}.json")

if not os.path.exists(example_path):
    raise FileNotFoundError(f"Missing output file: {example_path}")

with open(example_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Example file:", example_path)
print("scene:", data.get("scene"))
print("num_all_objects:", data.get("num_all_objects"))
print("num_valid_objects:", data.get("num_valid_objects"))
print("num_chunks:", data.get("num_chunks"))
print("num_clusters:", data.get("num_clusters"))
print("total_cluster_object_mentions:", data.get("total_cluster_object_mentions"))

print("\nchunk_config:")
print(json.dumps(data.get("chunk_config", {}), indent=2, ensure_ascii=False))

print("\nFirst 5 chunks summary:")
for chunk in data.get("chunks", [])[:5]:
    print(
        chunk["chunk_id"],
        "grid=",
        (chunk["grid_i"], chunk["grid_j"]),
        "raw_objects=",
        chunk["num_raw_objects"],
        "clusters=",
        chunk["num_clusters"],
    )

Example file: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/step1_gt/FloorPlan1.json
scene: FloorPlan1
num_all_objects: 77
num_valid_objects: 76
num_chunks: 9
num_clusters: 6
total_cluster_object_mentions: 78

chunk_config:
{
  "grid_nx": 3,
  "grid_nz": 3,
  "parent_completion_radius": 1.5,
  "child_completion_radius": 1.5,
  "max_children_per_anchor": 5,
  "near_graph_threshold": 0.8,
  "min_objects_per_cluster": 4,
  "max_objects_per_cluster": 15,
  "min_anchors_per_cluster": 1,
  "min_small_objects_per_cluster": 1,
  "max_clusters_per_chunk": 2,
  "keep_weak_clusters": false
}

First 5 chunks summary:
FloorPlan1_chunk_0_0 grid= (0, 0) raw_objects= 15 clusters= 1
FloorPlan1_chunk_0_1 grid= (0, 1) raw_objects= 12 clusters= 1
FloorPlan1_chunk_0_2 grid= (0, 2) raw_objects= 3 clusters= 0
FloorPlan1_chunk_1_0 grid= (1, 0) raw_objects= 18 clusters= 1
FloorPlan1_chunk_1_1 grid= (1, 1) raw_objects= 7 clusters= 1


In [ ]:
import os
import sys
import shutil
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

archive_dir = os.path.join(cfg.DATA_DIR, "archives")
os.makedirs(archive_dir, exist_ok=True)

zip_base = os.path.join(archive_dir, "step1_gt")

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=cfg.STEP1_OUTPUT_DIR,
)

print("Created archive on Google Drive:", zip_path)

Created archive on Google Drive: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/archives/step1_gt.zip
